In [1]:
from pathlib import Path
import base64
import json
import os

import pandas as pd
import requests

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import (
    DEV_LOCATION_ID_LIST,
    apply_polaris_token_to_spark,
    create_minio_spark_session,
    ensure_fresh_polaris_user_token,
)

In [2]:
import os
print("POLARIS_CLIENT_ID:", os.getenv("POLARIS_CLIENT_ID"))
print("POLARIS_CLIENT_SECRET exists:", bool(os.getenv("POLARIS_CLIENT_SECRET")))
print("POLARIS_REFRESH_TOKEN exists:", bool(os.getenv("POLARIS_REFRESH_TOKEN")))

POLARIS_CLIENT_ID: jupyterhub
POLARIS_CLIENT_SECRET exists: True
POLARIS_REFRESH_TOKEN exists: True


In [9]:
import base64
import json
import os
from datetime import datetime, timezone

def decode_jwt(token: str):
    if not token:
        return None
    parts = token.split(".")
    if len(parts) != 3:
        return {"error": "Not a JWT (expected 3 dot-separated parts)"}
    payload = parts[1] + "=" * (-len(parts[1]) % 4)
    claims = json.loads(base64.urlsafe_b64decode(payload.encode()))
    return claims

def pretty_claims(name: str, token: str):
    claims = decode_jwt(token)
    print(f"\n{name}:")
    if claims is None:
        print("  <missing>")
        return
    if "error" in claims:
        print(" ", claims["error"])
        return
    for ts_key in ["iat", "exp", "auth_time"]:
        if ts_key in claims:
            claims[f"{ts_key}_utc"] = datetime.fromtimestamp(
                claims[ts_key], tz=timezone.utc
            ).isoformat()
    print(json.dumps(claims, indent=2, sort_keys=True))

access_token = os.getenv("POLARIS_USER_TOKEN", "")
refresh_token = os.getenv("POLARIS_REFRESH_TOKEN", "")

pretty_claims("POLARIS_USER_TOKEN claims", access_token)


POLARIS_USER_TOKEN claims:
{
  "acr": "1",
  "allowed-origins": [
    "https://hub.teehr.local.app.garden"
  ],
  "aud": [
    "realm-management",
    "account"
  ],
  "auth_time": 1784820452,
  "auth_time_utc": "2026-07-23T15:27:32+00:00",
  "azp": "jupyterhub",
  "email": "admin@example.local",
  "email_verified": true,
  "exp": 1784820752,
  "exp_utc": "2026-07-23T15:32:32+00:00",
  "family_name": "Admin",
  "given_name": "Local",
  "groups": [
    "basic-user",
    "iceberg-catalog-admins",
    "iceberg-user",
    "jupyter-admin",
    "key-management-admin",
    "prefect-admin",
    "webapi-admin"
  ],
  "iat": 1784820452,
  "iat_utc": "2026-07-23T15:27:32+00:00",
  "iss": "https://auth.teehr.local.app.garden/realms/teehr",
  "jti": "ofrtac:3719a618-6e89-7a46-38fb-c9a93cddf030",
  "name": "Local Admin",
  "preferred_username": "admin",
  "realm_access": {
    "roles": [
      "iceberg-catalog-admin",
      "offline_access",
      "admin",
      "uma_authorization",
      "basic-us

In [10]:
claims = decode_jwt(os.getenv("POLARIS_USER_TOKEN", ""))
print("sub:", claims.get("sub"))
print("preferred_username:", claims.get("preferred_username"))
print("groups:", claims.get("groups"))
print("realm roles:", claims.get("realm_access", {}).get("roles", []))
print("exp:", claims.get("exp"))

sub: 8f4441da-7756-49e3-9142-64c299ed6c39
preferred_username: admin
groups: ['basic-user', 'iceberg-catalog-admins', 'iceberg-user', 'jupyter-admin', 'key-management-admin', 'prefect-admin', 'webapi-admin']
realm roles: ['iceberg-catalog-admin', 'offline_access', 'admin', 'uma_authorization', 'basic-user', 'default-roles-teehr', 'jupyter-user', 'iceberg-user']
exp: 1784820752


In [11]:
import time
from datetime import datetime, timezone

now = int(time.time())
print("now_epoch:", now)
print("now_utc:", datetime.fromtimestamp(now, tz=timezone.utc).isoformat())

now_epoch: 1784820789
now_utc: 2026-07-23T15:33:09+00:00


In [48]:
claims = decode_jwt(os.getenv("POLARIS_USER_TOKEN", ""))  # from previous helper
exp = int(claims.get("exp", 0))
now = int(time.time())
print("exp:", exp)
print("now:", now)
print("seconds_remaining:", exp - now)

exp: 1784825530
now: 1784825236
seconds_remaining: 294


In [47]:
polaris_user_token = os.environ.get("POLARIS_USER_TOKEN")
polaris_refresh_token = os.getenv("POLARIS_REFRESH_TOKEN", "")
polaris_username = os.getenv("POLARIS_USERNAME", os.getenv("JUPYTERHUB_USER", "admin"))
polaris_password = os.getenv("POLARIS_PASSWORD", "")
polaris_client_id = os.getenv("POLARIS_CLIENT_ID", "jupyterhub")
polaris_client_secret = os.getenv("POLARIS_CLIENT_SECRET")

if not polaris_user_token and not polaris_refresh_token and not polaris_password:
    raise RuntimeError(
        "No usable Polaris credentials found. Set POLARIS_USER_TOKEN or POLARIS_REFRESH_TOKEN in the session."
    )

polaris_user_token, polaris_refresh_token, token_renewed = ensure_fresh_polaris_user_token(
    current_token=polaris_user_token,
    username=polaris_username,
    password=polaris_password,
    client_id=polaris_client_id,
    client_secret=polaris_client_secret,
    refresh_token=polaris_refresh_token,
    refresh_window_seconds=120,
)
os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
if polaris_refresh_token:
    os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token

payload = polaris_user_token.split(".")[1]
payload += "=" * (-len(payload) % 4)
claims = json.loads(base64.urlsafe_b64decode(payload.encode()))
roles = sorted(claims.get("realm_access", {}).get("roles", []))

print("Token status:", "renewed" if token_renewed else "reused")
print("Refresh token available:", bool(polaris_refresh_token))
print("Token user:", claims.get("preferred_username"))
print("Token roles:", ", ".join(roles))
if "iceberg-catalog-admin" not in roles:
    print("WARNING: token is missing iceberg-catalog-admin; Polaris may reject writes")

trying to refresh token
eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJkMTJhN2FiYy04MzVkLTRiNzAtODgxMi02NjcxMjMzYjI1ODUifQ.eyJpYXQiOjE3ODQ4MjI1MzQsImp0aSI6IjI2ZjI5MmJiLWVkNGMtOWY4NS04MzRmLWEzNjNmOTQ3ZDYwZiIsImlzcyI6Imh0dHBzOi8vYXV0aC50ZWVoci5sb2NhbC5hcHAuZ2FyZGVuL3JlYWxtcy90ZWVociIsImF1ZCI6Imh0dHBzOi8vYXV0aC50ZWVoci5sb2NhbC5hcHAuZ2FyZGVuL3JlYWxtcy90ZWVociIsInN1YiI6IjhmNDQ0MWRhLTc3NTYtNDllMy05MTQyLTY0YzI5OWVkNmMzOSIsInR5cCI6Ik9mZmxpbmUiLCJhenAiOiJqdXB5dGVyaHViIiwic2lkIjoiRHh4VXR3ZDB4dTV4a2hTRURjc0llYnpkIiwic2NvcGUiOiJvcGVuaWQgYWNyIG9mZmxpbmVfYWNjZXNzIHJvbGVzIGJhc2ljIHByb2ZpbGUgZW1haWwgd2ViLW9yaWdpbnMiLCJhdWRfeCI6WyJyZWFsbS1tYW5hZ2VtZW50IiwiYWNjb3VudCJdfQ.VUSlhU7bpO6pRxlDE_RMNNpDv-HcLKZ0bThk-5ewvOoHGtae9QLxBhdqFCnZg7K_-bK8Tbpt9eKteInSImNwKA
{'grant_type': 'refresh_token', 'client_id': 'jupyterhub', 'refresh_token': 'eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJkMTJhN2FiYy04MzVkLTRiNzAtODgxMi02NjcxMjMzYjI1ODUifQ.eyJpYXQiOjE3ODQ4MjI1MzQsImp0aSI6IjI2ZjI5MmJiLWVkNGMtOWY4NS04MzRmLWE

In [42]:
import requests

realm = os.getenv("POLARIS_DEFAULT_REALM", "teehr")
remote_warehouse = os.getenv("REMOTE_WAREHOUSE_S3_PATH", "s3://warehouse/")
polaris_api_base = os.getenv("POLARIS_CATALOG_URI", "http://polaris:8181/api/catalog").rstrip("/")
catalog_config_url = f"{polaris_api_base}/v1/config"
mgmt_catalog_url = f"http://polaris:8181/api/management/v1/catalogs/{realm}"

# Refresh access token before probing Polaris or recreating Spark.
polaris_user_token, polaris_refresh_token, refreshed = ensure_fresh_polaris_user_token(
    current_token=polaris_user_token,
    username=polaris_username,
    password=polaris_password,
    client_id=polaris_client_id,
    client_secret=polaris_client_secret,
    refresh_token=polaris_refresh_token,
    refresh_window_seconds=120,
)
os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
if polaris_refresh_token:
    os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token
print("Cell 4 token status:", "renewed" if refreshed else "still fresh")

# Polaris REST config endpoint expects catalog identifier (realm), not S3 URI.
warehouse_for_config = remote_warehouse
if polaris_api_base.endswith("/api/catalog"):
    warehouse_for_config = realm

headers = {
    "Authorization": f"Bearer {polaris_user_token}",
    "X-Polaris-Realm": realm,
}

# Catalog config endpoint requires a warehouse query parameter.
catalog_resp = requests.get(
    catalog_config_url,
    headers=headers,
    params={"warehouse": warehouse_for_config},
    timeout=20,
 )
print("Catalog config API status:", catalog_resp.status_code)
print("Catalog config warehouse query:", warehouse_for_config)
if catalog_resp.status_code >= 400:
    print("Catalog config API body:", catalog_resp.text[:500])
catalog_resp.raise_for_status()

if "iceberg-catalog-admin" in roles:
    mgmt_resp = requests.get(mgmt_catalog_url, headers=headers, timeout=20)
    print("Management API status:", mgmt_resp.status_code)
    if mgmt_resp.status_code >= 400:
        print("Management API body:", mgmt_resp.text[:500])
    mgmt_resp.raise_for_status()
else:
    print(
        "Skipping Management API check: token does not include iceberg-catalog-admin. "
        "Catalog access is sufficient for non-admin Spark paths."
    )

print("Polaris access check passed.")

spark = create_minio_spark_session(
    polaris_token=polaris_user_token,
    force_recreate_session=True,
)


def refresh_polaris_token_for_spark(refresh_window_seconds: int = 120) -> bool:
    global polaris_user_token, polaris_refresh_token

    polaris_user_token, polaris_refresh_token, refreshed = ensure_fresh_polaris_user_token(
        current_token=polaris_user_token,
        username=polaris_username,
        password=polaris_password,
        client_id=polaris_client_id,
        client_secret=polaris_client_secret,
        refresh_token=polaris_refresh_token,
        refresh_window_seconds=refresh_window_seconds,
    )
    os.environ["POLARIS_USER_TOKEN"] = polaris_user_token
    if polaris_refresh_token:
        os.environ["POLARIS_REFRESH_TOKEN"] = polaris_refresh_token
    apply_polaris_token_to_spark(spark, polaris_user_token, catalog_name="iceberg", realm=realm)

    print("Spark token status:", "renewed" if refreshed else "still fresh")
    return refreshed


apply_polaris_token_to_spark(spark, polaris_user_token, catalog_name="iceberg", realm=realm)
print("Spark catalog namespace probe:")
spark.sql("SHOW NAMESPACES IN iceberg").show(truncate=False)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using user-provided AWS credentials
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...


trying to refresh token
eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJkMTJhN2FiYy04MzVkLTRiNzAtODgxMi02NjcxMjMzYjI1ODUifQ.eyJpYXQiOjE3ODQ4MjIzMjQsImp0aSI6IjJhNmQwZjQ3LWQ0YzItZjRiNS0xOTNjLTRhZGYzNjFiN2RiOSIsImlzcyI6Imh0dHBzOi8vYXV0aC50ZWVoci5sb2NhbC5hcHAuZ2FyZGVuL3JlYWxtcy90ZWVociIsImF1ZCI6Imh0dHBzOi8vYXV0aC50ZWVoci5sb2NhbC5hcHAuZ2FyZGVuL3JlYWxtcy90ZWVociIsInN1YiI6IjhmNDQ0MWRhLTc3NTYtNDllMy05MTQyLTY0YzI5OWVkNmMzOSIsInR5cCI6Ik9mZmxpbmUiLCJhenAiOiJqdXB5dGVyaHViIiwic2lkIjoiRHh4VXR3ZDB4dTV4a2hTRURjc0llYnpkIiwic2NvcGUiOiJvcGVuaWQgYWNyIG9mZmxpbmVfYWNjZXNzIHJvbGVzIGJhc2ljIHByb2ZpbGUgZW1haWwgd2ViLW9yaWdpbnMiLCJhdWRfeCI6WyJyZWFsbS1tYW5hZ2VtZW50IiwiYWNjb3VudCJdfQ.GTaneyP2O50XcRm0-BQc8gRuwoI_yJskKOGRFepFrtwlBu5JHz7EVi1sef5GzlyaMVr3VOC02VTGnMk3p1fpnQ
{'grant_type': 'refresh_token', 'client_id': 'jupyterhub', 'refresh_token': 'eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJkMTJhN2FiYy04MzVkLTRiNzAtODgxMi02NjcxMjMzYjI1ODUifQ.eyJpYXQiOjE3ODQ4MjIzMjQsImp0aSI6IjJhNmQwZjQ3LWQ0YzItZjRiNS0xOTNjLTR

INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


Spark catalog namespace probe:
+----------------+
|namespace       |
+----------------+
|teehr           |
|schema_evolution|
|restricted      |
|public          |
+----------------+



In [51]:
spark.stop()
spark = create_minio_spark_session(
    polaris_token=polaris_user_token,
    force_recreate_session=True,
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using user-provided AWS credentials
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [53]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [49]:
spark.conf.set("spark.sql.catalog.iceberg.token", polaris_user_token)
spark.conf.set("spark.sql.catalog.iceberg.rest.auth.oauth2.token", polaris_user_token)

In [54]:
ev.configurations.show()

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.


+--------------------+---------------+--------------------+--------------------+--------------------+----------+
|                name|timeseries_type|         description|          created_at|          updated_at|properties|
+--------------------+---------------+--------------------+--------------------+--------------------+----------+
|nrds_v22_cfenom_m...|      secondary|NRDS DataStream m...|2026-07-21 19:49:...|2026-07-21 19:49:...|      NULL|
|nrds_v22_cfenom_s...|      secondary|POC version of Da...|2026-07-21 19:49:...|2026-07-21 19:49:...|      NULL|
|nrds_v22_lstm0_me...|      secondary|NRDS DataStream m...|2026-07-21 19:49:...|2026-07-21 19:49:...|      NULL|
|nrds_v22_lstm0_sh...|      secondary|NRDS DataStream s...|2026-07-21 19:49:...|2026-07-21 19:49:...|      NULL|
|nwm30_analysis_as...|      secondary|Alaska NWM standa...|2026-07-21 19:49:...|2026-07-21 19:49:...|      NULL|
|nwm30_analysis_as...|      secondary|Alaska NWM extend...|2026-07-21 19:49:...|2026-07-21 19:49

In [ ]:
# spark.stop()